In [1]:
from spiral import Spiral

sp = Spiral(overrides={
    "keys_cache.enabled": "1",
    "keys_cache.memory_capacity_bytes": "1073741824",  # 1 GiB
    "keys_cache.disk_capacity_bytes": "0",
    # "fragments_cache.enabled": "1",
    # "fragments_cache.memory_capacity_bytes": "4294967296",  # 4 GiB
    # "fragments_cache.disk_capacity_bytes": "0",
    "manifests_cache.enabled": "1",
    "manifests_cache.memory_capacity_bytes": "0",
    "manifests_cache.disk_capacity_bytes": "1073741824",  # 1GiB
})

In [2]:
project = sp.project("enigma-spiral-poc-r2-2-754379")

In [3]:
project.list_tables()

[TableResource(id='table_26k2ou', project_id='enigma-spiral-poc-r2-2-754379', dataset='default', table='stim_events'),
 TableResource(id='table_53xdnx', project_id='enigma-spiral-poc-r2-2-754379', dataset='default', table='lfp_probe_metadata'),
 TableResource(id='table_57j3br', project_id='enigma-spiral-poc-r2-2-754379', dataset='default', table='vidtok_embeddings'),
 TableResource(id='table_7dcmj3', project_id='enigma-spiral-poc-r2-2-754379', dataset='default', table='session_reconstructed_video_metadata'),
 TableResource(id='table_9ntf2t', project_id='enigma-spiral-poc-r2-2-754379', dataset='default', table='vidtok_embeddings_patchidxs'),
 TableResource(id='table_hm8n18', project_id='enigma-spiral-poc-r2-2-754379', dataset='default', table='vjepa_embeddings_patchidxs'),
 TableResource(id='table_j56ky7', project_id='enigma-spiral-poc-r2-2-754379', dataset='default', table='unit_metadata'),
 TableResource(id='table_jvtb8e', project_id='enigma-spiral-poc-r2-2-754379', dataset='default',

In [3]:
tbl_session_metadata = project.table("session_reconstructed_video_metadata")
tbl_vidtok_embeddings = project.table("vidtok_embeddings_patchidxs")
tbl_vjepa_embeddings = project.table("vjepa_embeddings_patchidxs")
tbl_behavior_adc = project.table("behavior_adc")

2026-01-30T12:04:49.242807Z  INFO spiral_client: Initializing keys cache name="keys" path=/home/mbakovic/.cache/spiral memory_capacity=1073741824 disk_capacity=0
2026-01-30T12:04:49.243319Z  INFO foyer_storage::store: [store builder]: No I/O engine builder is provided, use `PsyncIoEngineConfig` with default parameters as default.
2026-01-30T12:04:49.243386Z  WARN foyer_storage::engine::block::engine: [block engine]: block-based object disk cache stable blocks count is too small, flusher [1] + clean block threshold [1] (default = reclaimers) is supposed to be much larger than the block count [0]
2026-01-30T12:04:49.243404Z  INFO foyer_storage::engine::block::recover: Recovers 0 blocks with data, 0 clean blocks, 0 total entries with max sequence as 0..
2026-01-30T12:04:49.243409Z  INFO foyer_storage::engine::block::recover: [recover] finish in 17.169µs
2026-01-30T12:04:49.243539Z  INFO spiral_client: Initializing manifests cache name="manifests" path=/home/mbakovic/.cache/spiral memory_c

In [5]:
tbl_vidtok_embeddings.schema()

Schema({session_id=utf8?, mean_frame_timestamp=i64?, modality=utf8?, patch_index=i64?, first_frame=i64?, last_frame=i64?, first_frame_timestamp=i64?, last_frame_timestamp=i64?, source_video_name=utf8?, embedding={embedding=list(f32?)?}?})

In [ ]:
sp.scan_keys(tbl_vidtok_embeddings, limit=10000).to_polars()

In [6]:
tbl_vjepa_embeddings.schema()

Schema({session_id=utf8?, mean_frame_timestamp=i64?, layer=i64?, patch_index=i64?, first_frame=i64?, last_frame=i64?, first_frame_timestamp=i64?, last_frame_timestamp=i64?, source_video_name=utf8?, embedding={embedding=list(f32?)?}?})

In [ ]:
sp.scan_keys(tbl_vjepa_embeddings, limit=10000).to_polars()

In [7]:
tbl_session_metadata.schema()

Schema({session_id=utf8?, timestamp=i64?, reconstruction_frame_number=i64?, bg_color=list(f64?)?, display=utf8?, displayed_movie=utf8?, displayed_movie_frame_number=f64?, source_movie=utf8?, source_movie_frame_number=f64?, trial_idx=i64?, fixation_dot={fixSpotColor=list(i16?)?, fixSpotLocation=list(i16?)?, fixSpotRingThickness=i64?, fixSpotSize=i64?}?})

In [5]:
tbl_behavior_adc.schema()

Schema({session_id=utf8?, timestamp=i64?, eye_x_px_offset_center=f64?, eye_y_px_offset_center=f64?, neuropixel_sync_in=f64?, photodiode=f64?, pupil_size_in=f64?, reward_input=f64?})

In [6]:
sp.scan_keys(tbl_behavior_adc, limit=1000).to_polars()

session_id,timestamp
str,i64
"""Goliath_2025-10-20_20-40-05""",28317294
"""Goliath_2025-10-20_20-40-05""",28317460
"""Goliath_2025-10-20_20-40-05""",28317627
"""Goliath_2025-10-20_20-40-05""",28317794
"""Goliath_2025-10-20_20-40-05""",28317960
…,…
"""Goliath_2025-10-20_20-40-05""",28483111
"""Goliath_2025-10-20_20-40-05""",28483277
"""Goliath_2025-10-20_20-40-05""",28483444


In [ ]:
sp.scan_keys(
    tbl_session_metadata,
    # Session for which embeddings are ingested
    where=tbl_session_metadata["session_id"] == "Goliath_2025-10-21_19-23-25",
    limit=1000,
).to_polars()

In [8]:
from spiral import Shard, KeyRange

def create_1second_shards(tbl, list_ranges) -> list[Shard]:
    return [
        Shard(KeyRange(
            begin=tbl.key(r["session_id"], r["timestamp"]),
            # Timestamps as in microseconds. Take a 1-second range.
            end=tbl.key(r["session_id"], r["timestamp"] + 1_000_000)
        ))
        for r in list_ranges
    ]

metadata_keys = sp.scan_keys(
    tbl_session_metadata,
    # Session for which embeddings are ingested
    where=tbl_session_metadata["session_id"] == "Goliath_2025-10-21_19-23-25",
).to_table().to_pylist()

session_shards = create_1second_shards(tbl_session_metadata, metadata_keys)
len(session_shards)

1293774

In [9]:
import numpy as np

# TODO(marko): This method is broken after flattening embeddings tables.
def stack_item(item):
    """
    Convert a single batch item to numpy arrays (most efficient version).
    Handles variable lengths: behavior can be 600 or 601, vidtok can have 7 or 8 embeddings.

    Args:
        item: Tuple of (behavior_batch, embeddings_batch)

    Returns:
        dict[str, np.ndarray]: Dictionary with 'behavior' and 'vidtok' arrays
    """
    behavior_batch = item[0]
    embeddings_batch = item[1]
    vjepa_batch = item[2]

    # Stack behavior data (600 or 601, 3)
    behavior_array = np.column_stack([
        behavior_batch["pupil_size_in"].to_numpy(zero_copy_only=False),
        behavior_batch["eye_x_px_offset_center"].to_numpy(zero_copy_only=False),
        behavior_batch["eye_y_px_offset_center"].to_numpy(zero_copy_only=False)
    ])

    # Stack vidtok embeddings
    tensor_column = embeddings_batch["tensor"]
    num_vidtoks = len(tensor_column)

    # Double flatten: list<list<float>> -> flat float array
    flattened_once = pa.ListArray.flatten(tensor_column)
    flattened_twice = pa.ListArray.flatten(flattened_once)

    # Zero-copy to numpy
    vidtok_flat = flattened_twice.to_numpy(zero_copy_only=False)

    # Infer number of rows per vidtok (should be 1590)
    num_vidtok_rows = len(vidtok_flat) // (num_vidtoks * 16)

    # Reshape and transpose: (num_vidtoks, num_vidtok_rows, 16) -> (num_vidtok_rows, 16, num_vidtoks)
    vidtok_array = vidtok_flat.reshape(num_vidtoks, num_vidtok_rows, 16).transpose(1, 2, 0)

    # Stack vjepa embeddings (15 rows of (33792, 24) -> (1408, 24, 24, 15))
    vjepa_tensor_column = vjepa_batch["tensor"]
    num_vjepa = len(vjepa_tensor_column)  # Should be 15

    # Double flatten: list<list<float>> -> flat float array
    vjepa_flattened_once = pa.ListArray.flatten(vjepa_tensor_column)
    vjepa_flattened_twice = pa.ListArray.flatten(vjepa_flattened_once)

    # Zero-copy to numpy
    vjepa_flat = vjepa_flattened_twice.to_numpy(zero_copy_only=False)

    # Reshape: (15 * 33792 * 24) -> (15, 33792, 24) -> (15, 1408, 24, 24) -> (1408, 24, 24, 15)
    vjepa_array = vjepa_flat.reshape(num_vjepa, 33792, 24).reshape(num_vjepa, 1408, 24, 24).transpose(1, 2, 3, 0)

    return {
        "behavior": behavior_array,  # Shape: (600 or 601, 3)
        "vidtok": vidtok_array,      # Shape: (1590, 16, 7 or 8)
        "vjepa": vjepa_array         # Shape: (1408, 24, 24, 15)
    }

In [10]:
# Open scan once.
vjepa_embeddings_scan = sp.scan(tbl_vjepa_embeddings["embedding"], where=tbl_vjepa_embeddings["layer"] == 8)

In [13]:
import tqdm

vjepa_embeddings_loader = vjepa_embeddings_scan.to_record_batches(shards=session_shards, batch_readahead=32)

# TODO(marko): Why 5 when sampling frequency is 15?
vjepa_rows_per_item = 24 * 24 * 5
for item in tqdm.tqdm(vjepa_embeddings_loader):
    if item.num_rows != vjepa_rows_per_item:
        print(f"Got unexpected number of rows: {item.num_rows}")
    pass

KeyboardInterrupt: 

In [11]:
# Open scan once.
vidtok_embeddings_scan = sp.scan(tbl_vidtok_embeddings["embedding"], where=tbl_vidtok_embeddings["modality"] == "rgb")

In [12]:
import tqdm

vidtok_embeddings_loader = vidtok_embeddings_scan.to_record_batches(shards=session_shards, batch_readahead=32)

# TODO(marko): We have had this off by 1 before. Something wrong with timestamps.
vidtok_rows_per_item1 = 1590 * 7
vidtok_rows_per_item2 = 1590 * 8
for item in tqdm.tqdm(vidtok_embeddings_loader):
    if item.num_rows != vidtok_rows_per_item1 and item.num_rows != vidtok_rows_per_item2:
        print(f"Got unexpected number of rows: {item.num_rows}")
    pass

KeyboardInterrupt: 

In [ ]:
import tqdm
import pyarrow as pa
from spiral import Sampler

def behavior_sampler_function(array: pa.Array) -> pa.Array:
    return pa.array([i % 10 == 0 for i in range(len(array))])

# Must go through sp.sample to apply the sampler
behavior_loader: pa.RecordBatchReader = sp.sample(
    tbl_behavior_adc[["pupil_size_in", "eye_x_px_offset_center", "eye_y_px_offset_center"]],
    sampler=Sampler(behavior_sampler_function),
    # TODO(marko): Must use different shards, session is "Goliath_2025-10-20_20-40-05"
    shards=session_shards,
    batch_readahead=32
)

# TODO(marko): We have had this off by 1 before. Something wrong with timestamps.
behavior_rows_per_item1 = 600
behavior_rows_per_item2 = 601
for item in tqdm.tqdm(behavior_loader):
    if item.num_rows != vidtok_rows_per_item1 and item.num_rows != vidtok_rows_per_item2:
        print(f"Got unexpected number of rows: {item.num_rows}")
    pass